# 检索对抗攻击优化器评估

本 Notebook 针对不同优化器配置（`eval_target` 中的别名）在投毒目标上的优化效果进行统计评估。

评估行为由 `configs/evaluation/optimizer.yml` 控制：

- `eval_cwe`：待评估的 CWE 类别列表。
- `eval_mode`：`per_cwe` 分别统计各 CWE；`mixed` 仅汇总为 `CWE-MIXED`；`both` 同时输出两者。
  - 当 `eval_cwe=["CWE-MIXED"]` 时，任何配置模式都自动降级为一次 mixed 评估。
- `eval_target`：字典，键为优化器配置别名，值为对应的投毒目标文件路径。

输出：Markdown 表格汇总各优化器在不同 CWE 分组下的指标，保存至 `logs/evaluation/optimizer/{YYYY-MM-DD-HH-MM}.md`。

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.utils.io import load_json, load_yaml
from src.utils.log import setup_logging
from src.evaluation.optimizer.metrics import (
    compute_optimizer_metrics,
    format_metric_value,
)

setup_logging()
print("Logging configured")

## 1. 加载配置

In [ ]:
config = load_yaml(str(PROJECT_ROOT / "configs/evaluation/optimizer.yml"))
eval_cfg = config.get("eval", {})

VALID_EVAL_MODES = {"per_cwe", "mixed", "both"}


def resolve_optimizer_scope(eval_cfg):
    """返回 (eval_cwe, 配置模式, 生效模式, 分组顺序)。"""
    if "eval_isolated" in eval_cfg:
        raise ValueError(
            "eval.eval_isolated 已移除；请使用 eval.eval_mode: per_cwe | mixed | both"
        )
    raw_cwe = eval_cfg.get("eval_cwe")
    if not isinstance(raw_cwe, (list, tuple)) or isinstance(raw_cwe, (str, bytes)):
        raise TypeError("eval.eval_cwe 必须是非空 CWE 字符串数组")
    if not raw_cwe:
        raise ValueError("eval.eval_cwe 不能为空")
    if any(not isinstance(cwe, str) or not cwe.strip() for cwe in raw_cwe):
        raise TypeError("eval.eval_cwe 中每一项都必须是非空字符串")
    cwe_list = list(dict.fromkeys(cwe.strip() for cwe in raw_cwe))

    configured_mode = str(eval_cfg.get("eval_mode", "")).strip().lower()
    if configured_mode not in VALID_EVAL_MODES:
        raise ValueError(
            "eval.eval_mode 必须是 per_cwe、mixed 或 both，"
            f"实际为 {eval_cfg.get('eval_mode')!r}"
        )
    if "CWE-MIXED" in cwe_list and cwe_list != ["CWE-MIXED"]:
        raise ValueError("eval.eval_cwe 不能同时包含 CWE-MIXED 与普通 CWE")

    effective_mode = "mixed" if cwe_list == ["CWE-MIXED"] else configured_mode
    if effective_mode == "per_cwe":
        group_order = list(cwe_list)
    elif effective_mode == "mixed":
        group_order = ["CWE-MIXED"]
    else:
        group_order = list(cwe_list) + ["CWE-MIXED"]
    return cwe_list, configured_mode, effective_mode, group_order


eval_cwe, configured_eval_mode, effective_eval_mode, group_order = resolve_optimizer_scope(eval_cfg)
if configured_eval_mode != effective_eval_mode:
    print(
        f"[WARN] eval_cwe={eval_cwe} 已是混合数据；eval_mode={configured_eval_mode} "
        "自动降级为 effective_eval_mode=mixed"
    )
eval_target = eval_cfg.get("eval_target", {})

print(f"eval_cwe: {eval_cwe}")
print(f"eval_mode: {configured_eval_mode}")
print(f"effective_eval_mode: {effective_eval_mode}")
print(f"group_order: {group_order}")
print(f"eval_target aliases: {list(eval_target.keys())}")

## 2. 加载投毒目标文件

按 `eval_target` 中定义的别名逐个加载投毒目标文件。文件格式须为 `{CWE: [entries...]}` 的字典形式。

In [ ]:
def load_target_data(path: str) -> dict:
    """加载投毒目标文件，支持相对路径（基于项目根目录）。"""
    abs_path = Path(path) if os.path.isabs(path) else PROJECT_ROOT / path
    data = load_json(str(abs_path))
    if not isinstance(data, dict):
        raise ValueError(f"Target file {path} is not a dict-formatted file (expected {{CWE: [entries...]}})")
    return data


target_data_by_alias = {}
for alias, path in eval_target.items():
    target_data_by_alias[alias] = load_target_data(path)
    print(f"  [{alias}] loaded from {path}")

print(f"\nTotal aliases loaded: {len(target_data_by_alias)}")

## 3. 构建 CWE 分组

根据 `eval_mode` 决定分组方式：

- `per_cwe`：为 `eval_cwe` 中每个 CWE 单独建组。
- `mixed`：将所有源 CWE 的样本按 `entity.id` 去重后合并为 `CWE-MIXED`。
- `both`：同时构造上述两类分组。

In [ ]:
def build_group_entries(data: dict, cwe_list: list, eval_mode: str) -> dict:
    """构建 CWE 分组，返回 {group_name: [entries...]}。"""
    groups = {}
    if eval_mode in {"per_cwe", "both"}:
        for cwe in cwe_list:
            groups[cwe] = list(data.get(cwe, []))
    if eval_mode in {"mixed", "both"}:
        mixed = []
        seen_ids = set()
        for cwe in cwe_list:
            for idx, entry in enumerate(data.get(cwe, [])):
                entity = entry.get("entity", {}) or {}
                raw_id = entity.get("id", f"__index_{idx}__")
                if raw_id in seen_ids:
                    continue
                seen_ids.add(raw_id)
                mixed.append(entry)
        groups["CWE-MIXED"] = mixed
    return groups


group_entries_by_alias = {}
for alias, data in target_data_by_alias.items():
    group_entries_by_alias[alias] = build_group_entries(data, eval_cwe, effective_eval_mode)

print(f"Group order: {group_order}")
for alias, groups in group_entries_by_alias.items():
    for grp, entries in groups.items():
        print(f"  [{alias}] {grp}: {len(entries)} total entries")

## 4. 计算评估指标

对每个优化器别名、每个 CWE 分组，调用 `compute_optimizer_metrics` 计算：

| 指标 | 说明 |
|------|------|
| total_samples | 已完成优化的样本总数 |
| total_time | 所有样本的总制作时间（秒） |
| avg_time | 单个样本的平均制作时间（秒） |
| avg_sim_improvement | 平均相似度提升 `(final_avg_sim - init_avg_sim)` |
| avg_per_iter_sim_improvement | 平均每轮相似度提升（使用实际迭代轮数） |
| avg_perturbation_chars | 扰动字符数的平均值 |
| avg_perturbation_ratio | 扰动字符占 poisoned_buggy_code 长度的平均比例 |

In [ ]:
metrics_by_alias = {}
for alias, groups in group_entries_by_alias.items():
    metrics_by_alias[alias] = {}
    for cwe, entries in groups.items():
        metrics = compute_optimizer_metrics(entries)
        metrics_by_alias[alias][cwe] = metrics
        print(f"[{alias}] {cwe}: optimized={metrics['total_samples']}, "
              f"avg_time={metrics['avg_time']:.2f}s, "
              f"avg_sim_improvement={metrics['avg_sim_improvement']:.4f}")

print("\nMetrics computation completed.")

## 5. 渲染 Markdown 报告并保存

In [ ]:
METRIC_NAMES = [
    "total_samples",
    "total_time",
    "avg_time",
    "avg_sim_improvement",
    "avg_per_iter_sim_improvement",
    "avg_perturbation_chars",
    "avg_perturbation_ratio",
]

METRIC_HEADERS = {
    "total_samples": "Total Samples",
    "total_time": "Total Time (s)",
    "avg_time": "Avg Time (s)",
    "avg_sim_improvement": "Avg Sim Improvement",
    "avg_per_iter_sim_improvement": "Avg Sim Improvement / Iter",
    "avg_perturbation_chars": "Avg Perturbation Chars",
    "avg_perturbation_ratio": "Avg Perturbation Ratio",
}


def render_group_table(cwe: str, alias_to_metrics: dict) -> str:
    """渲染单个 CWE 分组的 Markdown 表格。"""
    lines = [f"## {cwe}", ""]
    header = "| Optimizer Alias | " + " | ".join(METRIC_HEADERS[m] for m in METRIC_NAMES) + " |"
    lines.append(header)
    lines.append("|" + "|".join(["---"] * (len(METRIC_NAMES) + 1)) + "|")
    for alias in sorted(alias_to_metrics.keys()):
        metrics = alias_to_metrics[alias]
        values = [format_metric_value(metrics.get(m, 0.0), m) for m in METRIC_NAMES]
        lines.append(f"| {alias} | " + " | ".join(values) + " |")
    lines.append("")
    return "\n".join(lines)


timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M")
output_dir = PROJECT_ROOT / "logs/evaluation/optimizer"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{timestamp}.md"

md_lines = [
    "# Optimizer Evaluation Report",
    "",
    f"**Generated at:** {timestamp}",
    f"**Evaluation Mode:** {configured_eval_mode}",
    f"**Effective Evaluation Mode:** {effective_eval_mode}",
    "",
]

for cwe in group_order:
    alias_to_metrics = {
        alias: metrics_by_alias[alias].get(cwe, {})
        for alias in metrics_by_alias
    }
    md_lines.append(render_group_table(cwe, alias_to_metrics))

md_content = "\n".join(md_lines)
print(md_content)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(md_content)
print(f"\nReport saved to: {output_path}")